In [ ]:
import pandas as pd 

df= pd.read_csv('/home/manethunter/Documents/L-and-T-Assignment-Fake-News-Classification/datasets/WELFake_Dataset.csv')

df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [8]:
df = df.drop(columns=['Unnamed: 0'])

In [9]:
df.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [10]:
df['text'] = df['text'].fillna('')
df['title'] = df['title'].fillna('')


In [11]:


from sklearn.model_selection import train_test_split


df['combined'] = df['title'] + ' ' + df['text']
X = df[['combined']]   # ← only this column now   
y =  df['label']
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
X_train,X_val,y_train,y_val = train_test_split(X_train, y_train, test_size=0.2, random_state= 42)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)

X_train_tfidf = tf_idf.fit_transform(X_train['combined'])
X_test_tfidf = tf_idf.transform(X_test['combined'])
X_val_tfidf = tf_idf.transform(X_val['combined'])

In [13]:
# %%
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear')
lr_model.fit(X_train_tfidf, y_train)

# %%
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbose=100
)
lgb_model.fit(X_train_tfidf, y_train,
              eval_set=[(X_val_tfidf, y_val)],
              callbacks=[lgb.early_stopping(50)])   

/home/manethunter/.pyenv/versions/3.11.13/envs/myenv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 23761, number of negative: 22404
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.991745
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.990912
[LightGBM] [Debug] init for col-wise cost 52.965344 seconds, init for row-wise cost 55.094610 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 53.751862 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 4026049
[LightGBM] [Info] Number of data points in the train set: 46165, number of used features: 49867
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.514697 -> initscore=0.058806
[LightGBM] [Info] Start training from score 0.058806
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
Training until validation scores don't improve for 50 rounds
[Ligh

,n_estimators,1000
,random_state,42
,verbose,100
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [14]:
# %%
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

for name, model in [("Logistic Regression", lr_model), ("LightGBM", lgb_model)]:
    y_pred  = model.predict(X_test_tfidf)
    y_proba = model.predict_proba(X_test_tfidf)[:, 1]

    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba):.4f}")
    print(classification_report(y_test, y_pred))
    print()   

  Logistic Regression
Accuracy: 0.9620
ROC-AUC:  0.9938
              precision    recall  f1-score   support

           0       0.97      0.95      0.96      7089
           1       0.96      0.97      0.96      7338

    accuracy                           0.96     14427
   macro avg       0.96      0.96      0.96     14427
weighted avg       0.96      0.96      0.96     14427


  LightGBM
Accuracy: 0.9829
ROC-AUC:  0.9987
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7089
           1       0.98      0.99      0.98      7338

    accuracy                           0.98     14427
   macro avg       0.98      0.98      0.98     14427
weighted avg       0.98      0.98      0.98     14427




In [15]:
# %%
import joblib

# Save models + vectorizer
joblib.dump(lr_model, 'lr_model.pkl')
joblib.dump(lgb_model, 'lgb_model.pkl')
joblib.dump(tf_idf, 'tfidf_vectorizer.pkl')

print("Models saved.")

# %%
# Load models
lr_model_loaded = joblib.load('lr_model.pkl')
lgb_model_loaded = joblib.load('lgb_model.pkl')
tf_idf_loaded = joblib.load('tfidf_vectorizer.pkl')

# %%
# Preprocessing pipeline for new input
def predict_fake_news(title, text, model, vectorizer):
    combined = title + ' ' + text
    tfidf = vectorizer.transform([combined])
    pred = model.predict(tfidf)[0]
    proba = model.predict_proba(tfidf)[0]
    return pred, proba

# %%
# Test with user input
title = input("Enter title: ")
text  = input("Enter text: ")

print("\n--- Logistic Regression ---")
pred, proba = predict_fake_news(title, text, lr_model_loaded, tf_idf_loaded)
print(f"Prediction: {'FAKE' if pred == 1 else 'REAL'}")
print(f"Confidence: {max(proba)*100:.1f}%")

print("\n--- LightGBM ---")
pred, proba = predict_fake_news(title, text, lgb_model_loaded, tf_idf_loaded)
print(f"Prediction: {'FAKE' if pred == 1 else 'REAL'}")
print(f"Confidence: {max(proba)*100:.1f}%")   

Models saved.

--- Logistic Regression ---
Prediction: FAKE
Confidence: 87.6%

--- LightGBM ---
Prediction: FAKE
Confidence: 99.7%
